
# BTVN (Bản mở rộng): Transfer Learning & YOLO Video/Webcam

**Tác giả:** ChatGPT • **Ngày tạo:** 2025-10-16 16:09  
**Chủ đề:** Fine-tune ResNet/EfficientNet với `ImageFolder`, và demo **YOLOv8** trên video/webcam.

> Đây là **bổ sung** cho notebook trước. Bạn có thể dùng độc lập.  
> Mục tiêu: cung cấp **mẫu chạy được** (starter) cho dự án thực tế.


## 0) Cài đặt & import

In [ ]:

# (Tuỳ chọn) Cài đặt khi thiếu
# !pip install -U torch torchvision torchaudio ultralytics matplotlib scikit-learn

import os, sys, time, copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]



## 1) Transfer Learning (ImageFolder)

### Cấu trúc dữ liệu yêu cầu
```
data_dir/
    train/
        class_0/  img001.jpg, img002.jpg, ...
        class_1/  ...
        ...
    val/
        class_0/  ...
        class_1/  ...
        ...
```
> Nếu chưa có `val/`, bạn có thể dùng `torchvision.datasets.ImageFolder` + `random_split` để chia tạm.

### Tham số nhanh


In [ ]:

# TODO: sửa đường dẫn data_dir của bạn
data_dir = "/path/to/your_dataset"   # <-- THAY BẰNG THƯ MỤC CỦA BẠN

batch_size = 16
num_workers = 2
epochs = 5          # thử nhỏ trước (5-10), sau đó tăng
lr = 1e-3
use_mixup = False   # set True nếu muốn bật mixup đơn giản

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:

from torchvision.datasets import ImageFolder

train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")

train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(val_dir,   transform=val_tf)

num_classes = len(train_ds.classes)
class_names = train_ds.classes
print("Classes:", class_names)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False, num_workers=num_workers)


In [ ]:

import torch.nn.functional as F
def mixup_data(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, 1.0, y
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, lam, y_b


In [ ]:

def train_one_epoch(model, loader, optimizer, criterion, epoch, use_mixup=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (images, targets) in enumerate(loader):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        if use_mixup:
            inputs, y_a, lam, y_b = mixup_data(images, targets, alpha=0.4)
            outputs = model(inputs)
            loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
            preds = outputs.argmax(dim=1)
            # ước lượng accuracy thô (không chính xác với mixup, chỉ tham khảo)
            correct += (preds == targets).sum().item()
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / total, correct / total


In [ ]:

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds, all_tgts = [], []

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += images.size(0)

        all_preds.append(preds.cpu().numpy())
        all_tgts.append(targets.cpu().numpy())

    avg_loss = running_loss / total
    acc = correct / total
    all_preds = np.concatenate(all_preds)
    all_tgts = np.concatenate(all_tgts)
    return avg_loss, acc, all_preds, all_tgts


### 1.1) Chọn backbone: ResNet18 hoặc EfficientNet-B0

In [ ]:

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn
import torch.optim as optim

backbone = "resnet18"  # "resnet18" hoặc "efficientnet_b0"

if backbone == "resnet18":
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
elif backbone == "efficientnet_b0":
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
else:
    raise ValueError("Backbone không hợp lệ")

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)


### 1.2) Huấn luyện & đánh giá

In [ ]:

best_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(1, epochs+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch, use_mixup=use_mixup)
    val_loss, val_acc, val_preds, val_tgts = evaluate(model, val_loader, criterion)

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={val_loss:.4f} acc={val_acc:.4f}")

# Load best
model.load_state_dict(best_wts)
print(f"Best val acc: {best_acc:.4f}")


In [ ]:

# Báo cáo phân loại & Confusion Matrix
val_loss, val_acc, val_preds, val_tgts = evaluate(model, val_loader, criterion)
print(f"Validation acc: {val_acc:.4f}")
print(classification_report(val_tgts, val_preds, target_names=class_names))

cm = confusion_matrix(val_tgts, val_preds, labels=list(range(num_classes)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(xticks_rotation=45)
plt.title("Confusion Matrix (Validation)")
plt.tight_layout()
plt.show()


In [ ]:

# Lưu model tốt nhất
save_path = f"./fine_tuned_{backbone}_{len(class_names)}cls_best.pth"
torch.save(model.state_dict(), save_path)
print("Đã lưu:", save_path)


### 1.3) Suy luận 1 ảnh đơn lẻ

In [ ]:

from torchvision.transforms.functional import to_pil_image

def predict_image(model, img_path, size=224):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    tfm = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    x = tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
        cls = int(np.argmax(prob))
    return img, cls, prob[cls]

# Ví dụ: thay đường dẫn ảnh bên dưới
# test_img_path = "/path/to/your_dataset/val/class_0/xxx.jpg"
# img, cls, p = predict_image(model, test_img_path)
# plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis("off")
# plt.title(f"Pred: {class_names[cls]} ({p:.2f})")
# plt.show()



---
## 2) YOLOv8: Video/Webcam Demo

- Yêu cầu: `pip install ultralytics opencv-python`  
- **Webcam**: `source=0` (có thể cần quyền camera).  
- **Video file**: đặt `VIDEO_PATH` tới file `.mp4`/`.avi` của bạn.


In [ ]:

try:
    from ultralytics import YOLO
    import cv2
    have_ultralytics = True
except Exception as e:
    have_ultralytics = False
    print("Thiếu 'ultralytics' hoặc 'opencv-python'. Cài bằng:")
    print("  pip install ultralytics opencv-python")

if have_ultralytics:
    # CHỌN NGUỒN:
    USE_WEBCAM = False       # True -> webcam; False -> dùng file video
    VIDEO_PATH = "/path/to/video.mp4"  # thay bằng đường dẫn thực tế khi USE_WEBCAM=False

    model = YOLO("yolov8n.pt")  # tải lần đầu nếu chưa có

    if USE_WEBCAM:
        cap = cv2.VideoCapture(0)
    else:
        cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Không mở được nguồn video/webcam. Kiểm tra đường dẫn hoặc quyền camera.")
    else:
        window_name = "YOLOv8 — Press 'q' to quit"
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            results = model(frame, conf=0.25, verbose=False)
            annotated = results[0].plot()
            cv2.imshow(window_name, annotated)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()
